In [3]:
import pandas as pd
import numpy as np

# ---------- 1) Helpers ----------
def convert_to_seconds(df: pd.DataFrame, time_cols):
    """
    Convert given timedelta-like text columns to seconds as float in new *_s columns.
    Keeps originals. Missing/invalid -> NaN.
    """
    for c in time_cols:
        td = pd.to_timedelta(df[c], errors="coerce")
        df[f"{c}_s"] = td.dt.total_seconds()
    return df

def fill_missing_times_single(df: pd.DataFrame, lap_col: str, s1_col: str, s2_col: str, s3_col: str):
    """
    Stage 1: Fill exactly-one-missing among [lap, s1, s2, s3] using lap = s1 + s2 + s3.
    Leaves rows with 2+ nulls as-is.
    Clips tiny negatives to 0 and rounds to 4 decimals.
    """
    L, S1, S2, S3 = lap_col, s1_col, s2_col, s3_col

    mask_lap_missing = df[L].isna() & df[S1].notna() & df[S2].notna() & df[S3].notna()
    df.loc[mask_lap_missing, L] = df.loc[mask_lap_missing, [S1, S2, S3]].sum(axis=1)

    mask_s1_missing = df[S1].isna() & df[L].notna() & df[S2].notna() & df[S3].notna()
    df.loc[mask_s1_missing, S1] = df.loc[mask_s1_missing, L] - df.loc[mask_s1_missing, [S2, S3]].sum(axis=1)

    mask_s2_missing = df[S2].isna() & df[L].notna() & df[S1].notna() & df[S3].notna()
    df.loc[mask_s2_missing, S2] = df.loc[mask_s2_missing, L] - df.loc[mask_s2_missing, [S1, S3]].sum(axis=1)

    mask_s3_missing = df[S3].isna() & df[L].notna() & df[S1].notna() & df[S2].notna()
    df.loc[mask_s3_missing, S3] = df.loc[mask_s3_missing, L] - df.loc[mask_s3_missing, [S1, S2]].sum(axis=1)

    # Clip tiny negative artifacts to 0, round
    for c in [L, S1, S2, S3]:
        df[c] = df[c].mask(df[c] < 0, 0)
    df[[L, S1, S2, S3]] = df[[L, S1, S2, S3]].round(4)

    return {
        "filled_lap": int(mask_lap_missing.sum()),
        "filled_s1": int(mask_s1_missing.sum()),
        "filled_s2": int(mask_s2_missing.sum()),
        "filled_s3": int(mask_s3_missing.sum()),
    }

def impute_multimissing_by_group_mean(
    df: pd.DataFrame,
    group_cols = ("raceId","driverId","compound"),
    s_cols = ("sector1Time_s","sector2Time_s","sector3Time_s"),
    lap_col = "lapTime_s"
):
    """
    Stage 2: For remaining rows with >=1 sector missing (especially 2+),
    fill missing sector(s) using the mean of that sector within the group
    defined by (raceId, driverId, compound). Then, if all three sectors
    are available, set lapTime_s = s1 + s2 + s3.

    Only uses group means; if a group's mean is unavailable, leaves NaN.
    """
    S1, S2, S3 = s_cols
    L = lap_col

    # Compute group means for each sector
    gmean_s1 = df.groupby(list(group_cols))[S1].transform('mean')
    gmean_s2 = df.groupby(list(group_cols))[S2].transform('mean')
    gmean_s3 = df.groupby(list(group_cols))[S3].transform('mean')

    before_na = df[[S1,S2,S3]].isna().sum()

    # Fill sectors where missing using group mean for that sector
    mask_s1 = df[S1].isna() & gmean_s1.notna()
    df.loc[mask_s1, S1] = gmean_s1[mask_s1]

    mask_s2 = df[S2].isna() & gmean_s2.notna()
    df.loc[mask_s2, S2] = gmean_s2[mask_s2]

    mask_s3 = df[S3].isna() & gmean_s3.notna()
    df.loc[mask_s3, S3] = gmean_s3[mask_s3]

    # After sector fills, recompute lap when all three sectors exist
    mask_all_sectors = df[S1].notna() & df[S2].notna() & df[S3].notna()
    df.loc[mask_all_sectors, L] = (df.loc[mask_all_sectors, [S1, S2, S3]].sum(axis=1))

    # Clean up: clip and round
    for c in [S1, S2, S3, L]:
        df[c] = df[c].mask(df[c] < 0, 0)
    df[[S1, S2, S3, L]] = df[[S1, S2, S3, L]].round(4)

    after_na = df[[S1,S2,S3]].isna().sum()
    filled_counts = (before_na - after_na).to_dict()

    return {
        "filled_s1_by_group": int(filled_counts.get(S1, 0)),
        "filled_s2_by_group": int(filled_counts.get(S2, 0)),
        "filled_s3_by_group": int(filled_counts.get(S3, 0)),
        "laps_recomputed": int(mask_all_sectors.sum())
    }

# ---------- 2) Run on your file ----------
path = r"D:\Masters Study Abroad\Self Study\Personal Projects\F1\Descriptive Analytics\LapTimes_2025.csv"
out_path = r"D:\Masters Study Abroad\Self Study\Personal Projects\F1\Descriptive Analytics\LapTimes_Cleaned_2025.csv"

# Read as strings and normalize blanks -> NaN
df = pd.read_csv(path, dtype=str).replace({"": np.nan})

# Convert textual times to seconds
time_cols = ["lapTime", "sector1Time", "sector2Time", "sector3Time"]
df = convert_to_seconds(df, time_cols)

# Stage 1: fill exactly-one-missing via equation
stage1_stats = fill_missing_times_single(
    df,
    lap_col="lapTime_s",
    s1_col="sector1Time_s",
    s2_col="sector2Time_s",
    s3_col="sector3Time_s"
)

# Stage 2: for remaining gaps (esp. 2+ missing), impute from group mean (raceId, driverId, compound)
stage2_stats = impute_multimissing_by_group_mean(
    df,
    group_cols=("raceId","driverId","compound"),
    s_cols=("sector1Time_s","sector2Time_s","sector3Time_s"),
    lap_col="lapTime_s"
)

# Save cleaned dataset (keeps original text columns + *_s numeric columns)
df.to_csv(out_path, index=False)

print("Saved:", out_path)
print("Stage 1 (equation) filled:", stage1_stats)
print("Stage 2 (group mean) filled:", stage2_stats)


Saved: D:\Masters Study Abroad\Self Study\Personal Projects\F1\Descriptive Analytics\LapTimes_Cleaned_2025.csv
Stage 1 (equation) filled: {'filled_lap': 259, 'filled_s1': 269, 'filled_s2': 0, 'filled_s3': 0}
Stage 2 (group mean) filled: {'filled_s1_by_group': 33, 'filled_s2_by_group': 33, 'filled_s3_by_group': 36, 'laps_recomputed': 15456}
